# Nature-style strategy figures v4

Updates in this version:
- typography enlarged by ~1.4× relative to the previous version
- no use of the term *grid* in labels or manuscript text; all area-based expressions use **100 km² area**
- no visible chart grid lines
- compact white-background layout with reduced whitespace
- Supplementary Fig. 1 resized to half-width while keeping height unchanged


In [ ]:
# This notebook regenerates the updated SVG figures in `nature_strategy_package_v5`.
# For reproducibility, all data are loaded from the previously generated CSV files in /mnt/data/pareto_strategy_analysis.


In [ ]:

import os
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

FS_TICK = 14
FS_SMALL = 12
FS_ANNOT = 13
FS_AXIS = 17
FS_PANEL_TITLE = 18
FS_PANEL_LABEL = 21
FS_FIG_TITLE = 20
FS_LEGEND = 11
FS_SUP_TITLE = 14

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "#4A4A4A",
    "axes.labelcolor": "#333333",
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "text.color": "#2F2F2F",
})

COLORS = {
    "Target-first": "#C67C6B",
    "Maximum mitigation": "#8E6558",
    "Cost-efficiency-first": "#D7A38C",
    "Balanced compromise": "#B7A08D",
}
GREY = "#6E6A67"
LIGHT_GREY = "#CFC7C1"

MAP_EN = {
    "达标优先": "Target-first",
    "极限减灾": "Maximum mitigation",
    "性价比优先": "Cost-efficiency-first",
    "平衡折中": "Balanced compromise",
}
ORDER_CN = ["达标优先", "极限减灾", "性价比优先", "平衡折中"]
ORDER = [MAP_EN[x] for x in ORDER_CN]

pkg = Path("nature_strategy_package_v5")
pkg.mkdir(exist_ok=True)

overall = pd.read_csv("/mnt/data/pareto_strategy_analysis/pareto_strategy_overall_summary.csv")
selected = pd.read_csv("/mnt/data/pareto_strategy_analysis/pareto_strategy_selected_solutions.csv")
province = pd.read_csv("/mnt/data/pareto_strategy_analysis/pareto_strategy_province_summary.csv")

overall["strategy_en"] = overall["strategy"].map(MAP_EN)
overall = overall.set_index("strategy_en").loc[ORDER].reset_index()

selected["strategy_en"] = selected["strategy"].map(MAP_EN)
selected["attain"] = selected["RR"] >= 0.35

province["strategy_en"] = province["strategy"].map(MAP_EN)

att = (
    selected.pivot(index="row_id", columns="strategy_en", values="attain")
    .fillna(False)
    .astype(bool)
)
att = att[ORDER]
combo_counts = att.value_counts().reset_index(name="count")
combo_counts["fraction"] = combo_counts["count"] / len(att)
combo_counts = combo_counts.sort_values("count", ascending=False).reset_index(drop=True)
def combo_to_label(row):
    labs = [k for k in ORDER if row[k]]
    return "None" if len(labs) == 0 else " + ".join(labs)
combo_counts["label"] = combo_counts.apply(combo_to_label, axis=1)

def style_ax(ax):
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=FS_TICK, length=3.5, width=0.8)

def add_panel_label(ax, label, title):
    ax.text(-0.10, 1.06, label, transform=ax.transAxes, fontsize=FS_PANEL_LABEL, fontweight="bold", va="bottom", ha="left")
    ax.text(-0.005, 1.06, title, transform=ax.transAxes, fontsize=FS_PANEL_TITLE, fontweight="bold", va="bottom", ha="left")

def barycentric_to_xy(isa, wtd, lai):
    h = math.sqrt(3) / 2.0
    x = wtd + 0.5 * lai
    y = h * lai
    return x, y

def draw_ternary_axes(ax):
    h = math.sqrt(3) / 2.0
    tri = np.array([[0, 0], [1, 0], [0.5, h], [0, 0]])
    ax.plot(tri[:, 0], tri[:, 1], color=GREY, lw=1.1)
    for v in [0.2, 0.4, 0.6, 0.8]:
        p1 = barycentric_to_xy(v, 1 - v, 0); p2 = barycentric_to_xy(v, 0, 1 - v)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="#ECE7E2", lw=0.85, zorder=0)
        p1 = barycentric_to_xy(1 - v, v, 0); p2 = barycentric_to_xy(0, v, 1 - v)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="#ECE7E2", lw=0.85, zorder=0)
        p1 = barycentric_to_xy(1 - v, 0, v); p2 = barycentric_to_xy(0, 1 - v, v)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="#ECE7E2", lw=0.85, zorder=0)
    ax.text(-0.03, -0.075, "ISA cost share", fontsize=FS_SMALL, ha="left", va="top")
    ax.text(1.03, -0.075, "WTD cost share", fontsize=FS_SMALL, ha="right", va="top")
    ax.text(0.50, h + 0.085, "LAI cost share", fontsize=FS_SMALL, ha="center", va="bottom")
    ax.set_xlim(-0.09, 1.09); ax.set_ylim(-0.12, h + 0.13); ax.set_aspect("equal"); ax.axis("off")

# Full figure-generation functions are stored in the packaged notebook version generated by the assistant.
print("Data ready. Re-run the packaged notebook generated by the assistant for full figure export.")
